In [1]:
# Data Manipulation
import pandas as pd
import numpy as np
import pandas_ta as ta

# Data Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Gradient Boosting
from xgboost import XGBClassifier

# Model Validation
from sklearn import metrics

# Warnings
import warnings
warnings.filterwarnings('ignore')

NumPy: 2.2.6
Pandas: 3.0.3
Scikit-learn: 1.9.0
XGBoost: 3.3.0
pandas-ta: <module 'pandas_ta' from 'C:\\Users\\gab\\anaconda3\\envs\\bitcoin_ml\\Lib\\site-packages\\pandas_ta\\__init__.py'>


In [2]:
btc=pd.read_csv('bitcoin.csv')
btc.head()
btc.shape
btc.info()
btc.describe()
btc['Date']=pd.to_datetime(btc['Date'])
btc.drop('Adj Close',axis=1,inplace=True)

In [ ]:
# Check for duplicate rows
btc.duplicated().sum()

# Check for missing values
btc.isnull().sum()

# Number of unique values per column
btc.nunique()

In [3]:
import pandas_ta as ta
import numpy as np

# ===============================
# MOVING AVERAGES
# ===============================

btc['EMA5'] = ta.ema(btc['Close'], length=5)
btc['EMA10'] = ta.ema(btc['Close'], length=10)
btc['EMA20'] = ta.ema(btc['Close'], length=20)
btc['EMA50'] = ta.ema(btc['Close'], length=50)
btc['EMA100'] = ta.ema(btc['Close'], length=100)
btc['EMA200'] = ta.ema(btc['Close'], length=200)

btc['SMA10'] = ta.sma(btc['Close'], length=10)
btc['SMA20'] = ta.sma(btc['Close'], length=20)
btc['SMA50'] = ta.sma(btc['Close'], length=50)
btc['SMA100'] = ta.sma(btc['Close'], length=100)
btc['SMA200'] = ta.sma(btc['Close'], length=200)

btc['WMA20'] = ta.wma(btc['Close'], length=20)
btc['HMA20'] = ta.hma(btc['Close'], length=20)

# ===============================
# MOMENTUM
# ===============================

btc['RSI'] = ta.rsi(btc['Close'], length=14)

macd = ta.macd(btc['Close'])
btc = btc.join(macd)

btc['CCI'] = ta.cci(
    btc['High'],
    btc['Low'],
    btc['Close'],
    length=20
)

btc['ROC'] = ta.roc(btc['Close'], length=10)

btc['MOM'] = ta.mom(btc['Close'], length=10)

btc['WILLR'] = ta.willr(
    btc['High'],
    btc['Low'],
    btc['Close']
)

# ===============================
# TREND
# ===============================

adx = ta.adx(
    btc['High'],
    btc['Low'],
    btc['Close']
)

btc = btc.join(adx)

# ===============================
# VOLATILITY
# ===============================

btc['ATR'] = ta.atr(
    btc['High'],
    btc['Low'],
    btc['Close']
)

bb = ta.bbands(btc['Close'])

btc = btc.join(bb)

kc = ta.kc(
    btc['High'],
    btc['Low'],
    btc['Close']
)

btc = btc.join(kc)

dc = ta.donchian(
    btc['High'],
    btc['Low']
)

btc = btc.join(dc)

# ===============================
# VOLUME
# ===============================

btc['OBV'] = ta.obv(
    btc['Close'],
    btc['Volume']
)

btc['AD'] = ta.ad(
    btc['High'],
    btc['Low'],
    btc['Close'],
    btc['Volume']
)

btc['CMF'] = ta.cmf(
    btc['High'],
    btc['Low'],
    btc['Close'],
    btc['Volume']
)

# ===============================
# ENGINEERED FEATURES
# ===============================

btc['EMA20_EMA50_Ratio'] = btc['EMA20'] / btc['EMA50']

btc['Close_EMA20'] = btc['Close'] - btc['EMA20']

btc['ATR_Close'] = btc['ATR'] / btc['Close']

btc['Volume_OBV'] = btc['Volume'] * btc['OBV']

# ===============================
# LAG FEATURES
# ===============================

btc['Close_Lag1'] = btc['Close'].shift(1)
btc['Close_Lag2'] = btc['Close'].shift(2)
btc['Close_Lag3'] = btc['Close'].shift(3)
btc['Close_Lag5'] = btc['Close'].shift(5)

btc['Volume_Lag1'] = btc['Volume'].shift(1)
btc['Volume_Lag2'] = btc['Volume'].shift(2)

btc['RSI_Lag1'] = btc['RSI'].shift(1)

btc['MACD_Lag1'] = btc['MACD_12_26_9'].shift(1)

# ===============================
# DATE FEATURES
# ===============================

btc['DayOfWeek'] = btc['Date'].dt.dayofweek
btc['Month'] = btc['Date'].dt.month
btc['Quarter'] = btc['Date'].dt.quarter

In [4]:
btc = btc.loc[:, ~btc.columns.duplicated()]
btc.columns.duplicated().sum()

np.int64(0)

In [5]:
btc.dropna(inplace=True)
btc.reset_index(drop=True, inplace=True)

In [6]:
btc['Target'] = np.where(
    btc['Close'].shift(-1) > btc['Close'],
    1,
    0
)

btc = btc.iloc[:-1]

In [9]:

model4_features = [

    'Open',
    'High',
    'Low',
    'Close',
    'Volume',

    'EMA5',
    'EMA10',
    'EMA20',
    'EMA50',
    'EMA100',
    'EMA200',

    'SMA10',
    'SMA20',
    'SMA50',
    'SMA100',
    'SMA200',

    'WMA20',
    'HMA20',

    'RSI',

    'MACD_12_26_9',
    'MACDh_12_26_9',
    'MACDs_12_26_9',

    'CCI',

    'ROC',

    'MOM',

    'WILLR',

    'ADX_14',
    'DMP_14',
    'DMN_14'
]
model5_features = model4_features + [

    'ATR',

   'BBL_5_2.0_2.0',
    'BBM_5_2.0_2.0',
    'BBU_5_2.0_2.0',
'BBB_5_2.0_2.0',
'BBP_5_2.0_2.0',

    'KCLe_20_2',
    'KCBe_20_2',
    'KCUe_20_2',

    'DCL_20_20',
    'DCM_20_20',
    'DCU_20_20',

    'OBV',
    'AD',
    'CMF'
]

In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier


def train_models(feature_list):

    # Select Features
    x = btc[feature_list]

    # Select Target
    y = btc['Target']

    # Train-Test Split
    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        shuffle=False
    )

    # Scaling
    scaler = StandardScaler()

    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)

    # Models
    lr = LogisticRegression(
        C=0.01,
        max_iter=1000,
        solver='liblinear'
    )

    svm = SVC(
        C=10,
        kernel='poly',
        degree=4,
        probability=True
    )

    xgb = XGBClassifier(
        learning_rate=0.01,
        max_depth=3,
        n_estimators=50,
        eval_metric='logloss'
    )

    models = [
        ("Logistic Regression", lr),
        ("Support Vector Machine", svm),
        ("XGBoost", xgb)
    ]

    for name, model in models:

        model.fit(x_train, y_train)

        train_auc = roc_auc_score(
            y_train,
            model.predict_proba(x_train)[:,1]
        )

        test_auc = roc_auc_score(
            y_test,
            model.predict_proba(x_test)[:,1]
        )

        print("="*60)
        print(name)
        print("Training ROC-AUC :", round(train_auc,4))
        print("Testing ROC-AUC  :", round(test_auc,4))
train_models(model5_features)

Logistic Regression
Training ROC-AUC : 0.5604
Testing ROC-AUC  : 0.5844
Support Vector Machine
Training ROC-AUC : 0.7194
Testing ROC-AUC  : 0.5384
XGBoost
Training ROC-AUC : 0.6625
Testing ROC-AUC  : 0.4888
